# Test Urban Environment

## Import Libraries

In [ ]:
import sys
import os
from pathlib import Path

# Add project root to path (adjust if notebook is in a subfolder)
project_root = Path.cwd().parent  # if notebook is in experiments/ or similar
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


import warnings

# Suppress the specific future warning from torchrl
warnings.filterwarnings(
    "ignore", 
    category=FutureWarning, 
    module="torchrl.modules.mcts.scores"
)

import torch
from torchrl.envs import EnvBase
from torchrl.data import (
    Composite, 
    Unbounded, 
    Bounded,
    Stacked
    
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from tensordict import LazyStackedTensorDict, TensorDict, TensorDictBase
from tensordict.base import _default_is_leaf, _is_leaf_nontensor

import numpy as np

# from urbanmarl.envs.urbanmarl_env import UrbanEnv
from urbanmarl.envs.base_env import UrbanEnv

## Create Environment

In [ ]:
config = {
    "num_uavs": 3,
    "num_ues": 20,
    "area_size": (500, 500),
    "max_time_slots": 50,
    "max_horizontal_speed": 49.0,
    "max_vertical_speed": 12.0,
    "max_transmit_power": 5.0,
    "frequency_ghz": 29.0,
    "g2a_bandwidth": 10e6,
    "noise_figure_db": 7.0,
    "agents": ["agent_0", "agent_1", "agent_2"]
    # "agents": ["uav_0", "uav_1", 'ue_0']
}
num_envs = 2
scenario = "uav_navigation"
# scenario = "uav_ue_los"
# scenario = "coverage"
env = UrbanEnv(
    num_envs = num_envs, # batch_size=torch.Size([2])
    continuous_actions = True,
    seed=0,
    device=device,
    scenario=scenario,
    **config)

## Test the reset method

In [ ]:
# test output of reset method
tensordict = env.reset()
# tensordict.keys(True, True)

print("Shape of the reterned TensorDict:", tensordict.batch_size)
# print(f"rollout: shape: {rollout.shape}, len: {len(tensordict.shape)}")
for key in tensordict.keys(True, True):
    print(f"{key}, shape: {tensordict[key].shape}")

## Test reset_at method

In [ ]:
# test output of reset method when passing "_reset" in input tensordict to reset_at method
tensordict = env.reset()

t = torch.zeros((env.batch_size[0], 1), dtype=torch.bool, device=env.device)
t[0,0] = True
source = {"_reset": t}
r = TensorDict(
    source=source,
    batch_size=env.batch_size,
    device=env.device,
)
env.reset(tensordict=r)

## Test specs

In [ ]:
env.check_env_specs()

In [ ]:
env._has_dynamic_specs

In [ ]:
env.full_reward_spec_unbatched

In [ ]:
for key in env.specs.keys(True, True):
    print(key, env.specs[key].shape)

reward = env.scenario.reward(env, 'uav')
# reward = reward.mean(dim=1)
reward, reward.shapef

In [ ]:
env.specs

In [ ]:
print("action_spec:", env.full_action_spec)
print("reward_spec:", env.full_reward_spec)
print("done_spec:", env.full_done_spec)
print("observation_spec:", env.observation_spec)

## Test Rollout

For fun, let’s see what a simple random rollout looks like. You can call env.rollout(n_steps) and get an overview of what the environment inputs and outputs look like. Actions will automatically be drawn at random from the action spec domain.


In [ ]:
n_rollout_steps = 3
rollout = env.rollout(n_rollout_steps)

print("Shape of the rollout TensorDict:", rollout.batch_size)
print(f"rollout: shape: {rollout.shape}, len: {len(rollout.shape)}")
for key in rollout.keys(True, True):
    print(f"{key}, shape: {rollout[key].shape}")
    

In [ ]:
assert rollout["done"].shape == torch.Size((env.batch_size[0], n_rollout_steps, 1)), f"{rollout["done"].shape} not equal {torch.Size((env.batch_size[0], n_rollout_steps, 1))}"

## Test step

In [ ]:
action = env.action_spec.sample()
print(action.keys(True, True))
for group in env.group_map:
    print(action.get((group, 'action')))

In [ ]:
action = env.action_spec.sample()
new_td = tensordict.update(action)
next_td = env.step(new_td)

print(f"next_td: {next_td.shape}, len: {len(next_td.shape)}")
for key in next_td.keys(True, True):
    print(f"{key} shape: {next_td[key].shape}")

In [ ]:
next_td.keys(True, True, sort=True)
for group in env.group_map.keys():
    print(next_td.get(("next", group)).keys())
    assert "reward" in next_td.get(("next", group)).keys()

In [ ]:
for key in next_td.keys(True, True):
    print(f"{key}, size: {next_td[key].shape}")

## Check inheretence

In [ ]:
UrbanEnv.__mro__

In [ ]:
env.reset()

## Plot Urban Map Example

In [ ]:
env._env.plot(batch_idx = 1)

## Test 3D rendering

In [ ]:
img = env.scenario.render(env, mode='human')